# Experiment 13 – Recommendation System from Sales Data Using Deep Learning

### Additional Experiment – TensorFlow/Keras

**Aim:** To implement a simple recommendation system from sales data using a neural network in TensorFlow/Keras.

### Learning Objectives
- Understand the basic idea of a recommendation system.
- Represent customers and products as numerical inputs.
- Prepare sales data for a deep-learning model.
- Train a neural network to predict whether a customer is likely to purchase a product.
- Generate product recommendations from predicted purchase scores.


## 1. Theory

A **Recommendation System** suggests products, services, movies, or other items that a user may be interested in.

In this experiment, we use simple **sales/interaction data** containing customers, products, and whether a purchase occurred.

The neural network learns a relationship between customer-product interactions and the purchase outcome.

### Basic Flow

```text
Sales Data
    ↓
Data Preparation
    ↓
Encode Customer/Product
    ↓
Neural Network
    ↓
Purchase Probability
    ↓
Recommend Products
```

### Important Note

This is a simplified educational recommendation system. Real-world systems may use collaborative filtering, matrix factorization, embeddings, deep recommendation networks, or hybrid approaches.

In [ ]:
# Step 1: Import required libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 2. Create Sample Sales Data

For this laboratory experiment, we create a small sales dataset.

`purchase = 1` means the customer purchased the product.

`purchase = 0` means the customer did not purchase the product.

The data contains customer ID, product ID, product category, price, and purchase information.

In [ ]:
# Step 2: Create sample sales/interaction data
sales_data = pd.DataFrame({
    "customer_id": [1,1,1,2,2,2,3,3,3,4,4,4,5,5,5,6,6,6,
                    7,7,7,8,8,8,9,9,9,10,10,10],
    "product_id": [1,2,3,1,2,4,1,3,5,2,4,6,1,5,6,2,3,6,
                    1,4,5,2,5,6,3,4,6,1,5,6],
    "category":   [1,2,1,1,2,2,1,1,3,2,2,3,1,3,3,2,1,3,
                    1,2,3,2,3,3,1,2,3,1,3,3],
    "price":      [500,1200,700,500,1200,1800,500,700,2500,1200,1800,3000,
                   500,2500,3000,1200,700,3000,500,1800,2500,1200,2500,3000,
                   700,1800,3000,500,2500,3000],
    "purchase":   [1,1,0,1,0,1,1,1,0,1,1,0,1,0,1,0,1,1,
                   1,0,1,1,1,0,0,1,1,1,0,1]
})

print(sales_data.head())
print("\nDataset shape:", sales_data.shape)

## 3. Understand the Sales Data

Example:

```text
customer_id = 2
product_id  = 4
category    = 2
price       = 1800
purchase    = 1
```

This means customer 2 purchased product 4, which belongs to category 2 and costs 1800.

The model will learn patterns from many such customer-product interactions.

In [ ]:
# Step 3: Prepare input features and target
features = ["customer_id", "product_id", "category", "price"]
X = sales_data[features].astype("float32").values
y = sales_data["purchase"].astype("float32").values

# Normalize numerical inputs
normalizer = layers.Normalization()
normalizer.adapt(X)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

## 4. Build the Recommendation Model

The neural network receives customer and product-related information and predicts a value between 0 and 1.

- Close to **1** → likely to purchase
- Close to **0** → less likely to purchase

The final sigmoid layer is suitable for this binary prediction problem.

In [ ]:
# Step 4: Build a simple deep-learning recommendation model
model = keras.Sequential([
    layers.Input(shape=(len(features),)),
    normalizer,
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the Model

The dataset is split into training and validation data. The model learns from the training examples and uses the validation set to monitor performance.

In [ ]:
# Step 5: Train the model
history = model.fit(
    X,
    y,
    validation_split=0.2,
    epochs=50,
    batch_size=4,
    verbose=0
)

print("Training completed.")

In [ ]:
# Step 6: Plot training accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Recommendation Model Accuracy")
plt.legend()
plt.show()

In [ ]:
# Step 7: Evaluate the model
loss, accuracy = model.evaluate(X, y, verbose=0)

print("Loss:", round(float(loss), 4))
print("Accuracy:", round(float(accuracy * 100), 2), "%")

## 6. Predict Purchase Probability

Let us consider a customer and several products. The model calculates a purchase probability for each customer-product combination.

In [ ]:
# Step 8: Predict purchase probabilities for all sales records
sales_data["purchase_probability"] = model.predict(X, verbose=0).flatten()

print(sales_data[["customer_id", "product_id", "price", "purchase", "purchase_probability"]].head(10))

## 7. Generate Recommendations for a Customer

We create candidate products for one customer and rank them according to predicted purchase probability.

Here we recommend products with the highest predicted scores.

In [ ]:
# Step 9: Generate recommendations for Customer 1
customer_id = 1

products = pd.DataFrame({
    "customer_id": [customer_id] * 6,
    "product_id": [1, 2, 3, 4, 5, 6],
    "category": [1, 2, 1, 2, 3, 3],
    "price": [500, 1200, 700, 1800, 2500, 3000]
})

candidate_X = products[features].astype("float32").values
products["score"] = model.predict(candidate_X, verbose=0).flatten()

recommendations = products.sort_values("score", ascending=False)

print("Recommended products for Customer", customer_id)
print(recommendations[["product_id", "category", "price", "score"]].to_string(index=False))

In [ ]:
# Step 10: Display the Top-3 recommendations
TOP_N = 3

top_recommendations = recommendations.head(TOP_N)

print(f"Top {TOP_N} Recommended Products for Customer {customer_id}:\n")
for _, row in top_recommendations.iterrows():
    print(
        f"Product {int(row['product_id'])} | "
        f"Category {int(row['category'])} | "
        f"Price ₹{row['price']:.0f} | "
        f"Score {row['score']:.3f}"
    )

In [ ]:
# Step 11: Visualize recommendation scores
plt.figure(figsize=(8, 5))
plt.bar(
    recommendations["product_id"].astype(str),
    recommendations["score"]
)
plt.xlabel("Product ID")
plt.ylabel("Predicted Purchase Probability")
plt.title(f"Product Recommendation Scores – Customer {customer_id}")
plt.show()

## 8. Student Practice

Perform the following activities:

1. Generate recommendations for Customer 2.
2. Generate recommendations for Customer 5.
3. Change the number of hidden layers and compare the accuracy.
4. Change the number of neurons from 32 to 64.
5. Add another feature such as customer age or product rating.
6. Change `TOP_N` from 3 to 5.
7. Explain why products with higher predicted probability are recommended first.

### Observation Table

| Customer | Top Recommended Product | Score |
|---|---|---|
| Customer 1 | __________ | ______ |
| Customer 2 | __________ | ______ |
| Customer 5 | __________ | ______ |

## 9. Result

Thus, a simple **recommendation system from sales data using a deep-learning model** was successfully implemented using TensorFlow/Keras. The neural network learned from customer-product sales interactions and generated ranked product recommendations using predicted purchase probabilities.

## Viva Questions

1. What is a recommendation system?
2. What type of data is used in this experiment?
3. What does `purchase = 1` mean?
4. Why is normalization used?
5. Why is sigmoid used in the output layer?
6. What is purchase probability?
7. How are products ranked for recommendation?
8. What is the difference between recommendation and classification?
9. Name two real-world recommendation systems.
10. What are the limitations of this simple recommendation model?

## 10. Conclusion

In this experiment, students learned how sales/interaction data can be used to build a basic deep-learning recommendation system. The model predicts customer-product purchase likelihood and uses these predictions to recommend products.